# Neural Brush Renderer Training

This notebook trains a neural network to imitate a potentially slower and non-differentiable brush renderer. The trained model enables gradient-based optimization of brush stroke parameters (check `paint.ipynb` notebook for more details).

This notebook is fully customizable. You can plug in your own implementations of the base classes (`Brush`, `DifferentiableBrush`, `BrushStrokeLoss`) to extend functionality or use any of the existing `Brush` implementations.

## Quick Start

Jump to the **Configuration** section to set up your training run, or simply "Run All" with default parameters that would train `GeneralNeuralRenderer` to mimic `TextureBrush` resembling oil-brush strokes.

## Outputs

- **Trained model:** `neural_renderer.pt`
- **Checkpoints:** `checkpoints/`
- **TensorBoard logs:** `tensorboard_logs/`

To view logs during or after training:
```bash
tensorboard --logdir=tensorboard_logs --load_fast=False
```

The logs include visualizations of durations, losses, and rendered brush strokes showing the entire training progression.
Note that you can also view them while the training is still in progress.

## Memory management

You can control memory requirements and speed of the training via following parameters:
* **`batch_size`** - batch size used during one optimizer step.
* **`train_step_samples`** - number of random samples that are generated at once at the
  beginning of each training step. Increasing number of `train_step_samples`, and proportionally lowering `train_steps` would lead to the same number of optimizer steps, but it would slow down training a bit. By default, imitator runs one epoch over these in each training step, but this can be increased via `train_step_repeats` parameter.

Pass them to the `Imitator` via `imitator_kwargs` parameter of the `train_and_log` function.

With the default configuration you can run training smoothly on the free Kaggle's NVIDIA P100 instance (16 GB of memory). The training of more complex brushes takes approximately 1.5h. However, some fast brushes like `RectangleBrush` can be as fast as 15 minutes.

## Implementation Details

The core component of this notebook is the `Imitator` class (from `painting.imitate`). It trains a `DifferentiableBrush` (neural network) to match the outputs of a target `Brush` by:

1. Sampling random brush parameters
2. Rendering target strokes using the real brush
3. Training the neural network to reproduce those strokes

The `train_and_log` helper function wraps `Imitator` with TensorBoard logging, checkpointing, and validation. Each training step calls `imitator.run_one_train_step()`, which handles the sampling, rendering, and backpropagation internally.

In [ ]:
import base64
import io
import re
from collections import defaultdict
from pathlib import Path
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import requests
import torch
import torchvision.transforms.functional as TF
from PIL import Image
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from torch.utils.tensorboard import SummaryWriter
from torchvision.utils import make_grid
from tqdm.notebook import tqdm

from painting.brushes import Brush, TextureBrush, DifferentiableBrush, GeneralNeuralRenderer
from painting.imitator import Imitator

In [ ]:
def _load_image_from_source(source: str) -> Image.Image:
    """Load PIL Image from local path, URL, or base64 string."""
    if source.startswith("data:"):
        match = re.match(r"data:[^;]+;base64,(.+)", source)
        if match:
            image_bytes = base64.b64decode(match.group(1))
            return Image.open(io.BytesIO(image_bytes))
        raise ValueError("Invalid data URI format")

    parsed = urlparse(source)
    if parsed.scheme in ("http", "https"):
        response = requests.get(source, timeout=30)
        response.raise_for_status()
        return Image.open(io.BytesIO(response.content))

    if len(source) > 300 and re.match(r"^[A-Za-z0-9+/=]+$", source):
        try:
            image_bytes = base64.b64decode(source)
            return Image.open(io.BytesIO(image_bytes))
        except Exception:
            pass

    return Image.open(source)


def load_texture(source: str, device: str):
    """Load texture as grayscale tensor [1, H, W] in [0, 1] range."""
    img = _load_image_from_source(source).convert("L")
    tensor = TF.pil_to_tensor(img).float() / 255.0
    return tensor.to(device)

In [ ]:
def train_and_log(
    brush: Brush,
    model: DifferentiableBrush,
    *,
    file_to_save: str = "neural_renderer.pt",
    train_steps: int = 1250,
    imitator_kwargs: dict | None = None,
    val_freq: int = 100,
    val_rand_samples: int = 256,
    checkpoint_freq: int = 100,
    device: str = "cpu",
    log_dir: str = "tensorboard_logs",
    checkpoint_dir: str = "checkpoints",
    checkpoint_to_resume_from: str | None = None,
):
    """Train neural renderer with TensorBoard logging and checkpointing."""
    if imitator_kwargs is None:
        imitator_kwargs = {}

    model.train()

    file_to_save = Path(file_to_save)
    file_to_save.parent.mkdir(parents=True, exist_ok=True)

    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    logger = SummaryWriter(log_dir)

    imitator = Imitator(
        model=model,
        brush=brush,
        device=device,
        **imitator_kwargs,
    )

    if checkpoint_to_resume_from is not None:
        imitator.load_state(checkpoint_to_resume_from)

    step_counters = defaultdict(lambda: 0)

    for step in tqdm(range(train_steps), desc="Training"):
        stats = imitator.run_one_train_step()

        stats_str = ' | '.join(
            f'{k}: {v}'
            for k, v in stats.items() if k not in ('batch_loss', 'batch_duration')
        )
        print(f"Step {step + 1}/{train_steps}; {stats_str}")

        for key, values in stats.items():
            for v in values:
                logger.add_scalar(f"Train_Stats/{key}", v, step_counters[key])
                step_counters[key] += 1

        # Validation step.
        if (step + 1) % val_freq == 0:
            with torch.no_grad():
                imitator.model.eval()

                brush_params = torch.rand((val_rand_samples, brush.brush_params_count), device=device)
                foreground_true, alpha_mask_true = imitator.brush.render_brush_stroke(brush_params)
                foreground_pred, alpha_mask_pred = imitator.model.render_brush_stroke(brush_params)

                val_loss = imitator.loss_fn(
                    foreground_pred=foreground_pred,
                    alpha_mask_pred=alpha_mask_pred,
                    foreground_target=foreground_true,
                    alpha_mask_target=alpha_mask_true,
                ).mean().item()
                logger.add_scalar("Val_Stats/val_loss_with_rand_samples", val_loss, step)

                imitator.model.train()

            # Save snapshot of the first 10 predictions.
            for i in range(min(foreground_pred.shape[0], 10)):
                image = make_grid(
                    tensor=[
                        # Logger expects values in range [0, 1] that's why we're normalizing it here.
                        foreground_pred[i] / 255,
                        foreground_true[i] / 255,
                        alpha_mask_pred[i].expand(3, -1, -1),
                        alpha_mask_true[i].expand(3, -1, -1),
                    ],
                    nrow=2,
                    padding=4,
                    pad_value=0.95,
                )
                logger.add_image(f"Val_Images/rand_{i}", image, step)

        if (step + 1) % checkpoint_freq == 0:
            imitator.save_state(checkpoint_dir / f"step_{step}.pt")

    torch.save(imitator.model.state_dict(), file_to_save)
    logger.close()

# Configuration
Define your brush to mimic, differentiable brush, imitator parameters and run training. There's also default behaviour
which uses TextureBrush and GeneralNeuralRenderer with ModulatedPixelShuffleNet. Default canvas size is 128. Note that
this only affects canvas size that's used during optimization of the brush parameters. Once you have them optimized, you
can use arbitrary canvas_size to render them with your Brush.

In [ ]:
# === Device & Canvas ===
device = "cuda" if torch.cuda.is_available() else "cpu"

# Most of the networks have fixed output size of 128. There's also light version of the networks with
# output size 32, but they didn't work that well with painting.
# This needs to be compatible with chosen model.
canvas_size = 128

# === Output Paths ===
model_save_path = "neural_renderer.pt"
tensorboard_log_dir = "tensorboard_logs"
checkpoint_dir = "checkpoints"

# === Training Hyperparameters ===
train_steps = 1250  # Total training steps
val_freq = 100  # Validate every N steps
checkpoint_freq = 100  # Save checkpoint every N steps
val_samples = 256  # Samples for validation
checkpoint_to_resume = None  # Path to resume from, or None if this is the first run

# === Brush & Model ===

# Define brush. Must inherit from the Brush class. You need to always specify device and canvas_size.
brush = None

# Fallback to TextureBrush by default.
if brush is None:
    brush = TextureBrush(
        large_vertical=load_texture("./assets/textures/large_vertical_brush.png", device),
        large_horizontal=load_texture("./assets/textures/large_horizontal_brush.png", device),
        small_vertical=load_texture("./assets/textures/small_vertical_brush.png", device),
        small_horizontal=load_texture("./assets/textures/small_horizontal_brush.png", device),
        canvas_size=canvas_size,
        device=device,
    )

# Define model. Must inherit from DifferentiableBrush class. There's a GeneralNeuralRenderer, which
# supports any NN which outputs [B, 4, canvas_size, canvas_size] tensors.
# The channel dimension represent RGBA channels.
model = None

# Fallback to GeneralNeuralRenderer with ModulatedPixelShuffleNet by default.
if model is None:
    model = GeneralNeuralRenderer(
        brush_params_count=brush.brush_params_count,
        canvas_size=canvas_size,
        device=device,
    )

# Training

In [ ]:
train_and_log(
    brush,
    model,
    file_to_save=model_save_path,
    train_steps=train_steps,
    val_freq=val_freq,
    val_rand_samples=val_samples,
    checkpoint_freq=checkpoint_freq,
    device=device,
    log_dir=tensorboard_log_dir,
    checkpoint_dir=checkpoint_dir,
    checkpoint_to_resume_from=checkpoint_to_resume,

    # Optionally specify any other imitator parameters.
    # If you would like to lower memory requirements, set batch_size and train_step_samples
    # parameters to smaller number.
    # imitator_kwargs=None,
)

## Visualizations
Note that these can be also looked directly on the TensorBoard.

In [ ]:
ea = EventAccumulator(tensorboard_log_dir, size_guidance={"scalars": 0})
ea.Reload()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

metrics = [
    ("Train_Stats/epoch_avg_loss", 0, "Training Loss"),
    ("Val_Stats/val_loss_with_rand_samples", 1, "Validation Loss")
]

for tag, col, title in metrics:
    if tag in ea.Tags().get("scalars", []):
        events = ea.Scalars(tag)
        x = [e.step for e in events]
        y = [e.value for e in events]

        if col == 1:
            axes[col].plot(x, y, marker='o', linestyle='-')
        else:
            axes[col].plot(x, y)

        axes[col].set_title(title)
        axes[col].set_xlabel("Step")
        axes[col].set_ylabel("Loss")
        axes[col].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Compare neural renderer vs real brush outputs on random strokes.
n_samples = 5

model.load_state_dict(torch.load(model_save_path, map_location=device, weights_only=True))
model.eval()

with torch.no_grad():
    brush_params = torch.rand((n_samples, brush.brush_params_count), device=device)

    fg_real, alpha_real = brush.render_brush_stroke(brush_params)
    fg_neural, alpha_neural = model.render_brush_stroke(brush_params)

fig, axes = plt.subplots(1, n_samples, figsize=(3 * n_samples, 3))

for i in range(n_samples):
    grid = make_grid(
        tensor=[
            fg_neural[i] / 255,
            fg_real[i] / 255,
            alpha_neural[i].expand(3, -1, -1),
            alpha_real[i].expand(3, -1, -1),
        ],
        nrow=2,
        padding=4,
        pad_value=0.95,
    )
    axes[i].imshow(grid.permute(1, 2, 0).cpu().clamp(0, 1))
    axes[i].axis("off")

fig.suptitle("Each cell: Pred FG | Real FG (top), Pred alpha mask | Real alpha mask (bottom)", y=0.02)
plt.tight_layout()
plt.show()